In [1]:
import pandas as pd
import numpy as np
import re
import string
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv(
    "spam.csv",
    encoding="latin-1"
)

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
df = df[['v1', 'v2']]

df.columns = [
    'label',
    'message'
]

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
print(df.shape)

df.info()

df['label'].value_counts()

(5572, 2)
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 541.4 KB


label
ham     4825
spam     747
Name: count, dtype: int64

In [5]:
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r'http\\S+',
        '',
        text
    )

    text = re.sub(
        r'www\\S+',
        '',
        text
    )

    text = re.sub(
        r'\\d+',
        '',
        text
    )

    text = text.translate(
        str.maketrans(
            '',
            '',
            string.punctuation
        )
    )

    return text

In [7]:
df['message'] = df['message'].apply(
    clean_text
)

df.head()

,label,message
0,0,go until jurong point crazy available only in ...
1,0,ok lar joking wif u oni
2,1,free entry in 2 a wkly comp to win fa cup fina...
3,0,u dun say so early hor u c already then say
4,0,nah i dont think he goes to usf he lives aroun...


In [8]:
X = df['message']

y = df['label']

In [9]:
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(X)

print(X.shape)

(5572, 9492)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(4457, 9492)
(1115, 9492)


In [11]:
model = MultinomialNB()

model.fit(
    X_train,
    y_train
)

print("Training Completed")

Training Completed


In [12]:
y_pred = model.predict(
    X_test
)

In [13]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    "Accuracy:",
    round(
        accuracy * 100,
        2
    ),
    "%"
)

Accuracy: 97.85 %


In [14]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.92      0.92      0.92       150

    accuracy                           0.98      1115
   macro avg       0.95      0.95      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [15]:
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

[[953  12]
 [ 12 138]]


In [16]:
pickle.dump(
    model,
    open(
        "spam_model.pkl",
        "wb"
    )
)

pickle.dump(
    vectorizer,
    open(
        "vectorizer.pkl",
        "wb"
    )
)

print("Model Saved Successfully")

Model Saved Successfully


In [17]:
loaded_model = pickle.load(
    open(
        "spam_model.pkl",
        "rb"
    )
)

loaded_vectorizer = pickle.load(
    open(
        "vectorizer.pkl",
        "rb"
    )
)

print("Model Loaded")

Model Loaded


In [18]:
email = [
    "Congratulations! You won a free iPhone. Click now to claim your prize."
]

email_vector = loaded_vectorizer.transform(
    email
)

prediction = loaded_model.predict(
    email_vector
)

if prediction[0] == 1:

    print("Spam Email")

else:

    print("Not Spam")

Spam Email


In [19]:
email = [
    "Hi Sakshi, your interview is scheduled tomorrow at 10 AM."
]

email_vector = loaded_vectorizer.transform(
    email
)

prediction = loaded_model.predict(
    email_vector
)

if prediction[0] == 1:

    print("Spam Email")

else:

    print("Not Spam")

Not Spam
